In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_112552\3336491994.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_112552\3336491994.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_112552\3336491994.py:8: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_112552\3336491994.py:8: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_pol

In [2]:
# 1.加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，用MCP获取工具

# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        },
        "12306-mcp": {
          "transport": "streamable_http",
          "url": "https://mcp.api-inference.modelscope.net/7e7536a1985041/mcp"
        }
    }
)
# 3.2.用client拉取tool
tools = await client.get_tools()

# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.invoke调用Agent， 传入的Dict，里面有messages
response = await agent.ainvoke(
    {"messages": [HumanMessage("帮我查一下下周四从北京到南京的高铁")]}
)

print(response["messages"][-1].content)

已经帮您查到 **下周四（2026-09-17）从北京到南京的高铁**，车次非常多，以下是主要信息（均为北京南/北京 → 南京南）：

## 早班车（06:00–09:00）
| 车次 | 出发→到达 | 历时 | 二等座 | 一等座 | 商务座 |
|------|-----------|------|--------|--------|--------|
| G547 | 北京南 06:18 → 南京南 10:48 | 04:30 | 463元 | 777元 | 余15张 1734元 |
| **G1** | 北京南 06:30 → 南京南 10:13 | 03:43 | 532元 | 851元 | 余2张 1861元 |
| G3 | 北京南 06:52 → 南京南 10:17 | 03:25 | 532元 | 851元 | 无票 |
| G815 | 北京南 07:04 → 南京南 11:33 | 04:29 | 481元 | 808元 | 无票 |
| G37 | 北京南 07:52 → 南京南 11:15 | 03:23 | 532元 | 余9张 851元 | 无票 |
| G39 | 北京南 07:56 → 南京南 11:19 | 03:23 | 532元 | 851元 | 无票 |

## 上午车次（09:00–12:00）
| 车次 | 出发→到达 | 历时 | 二等座 | 一等座 |
|------|-----------|------|--------|--------|
| **G43** | 北京南 09:03 → 南京南 12:29 | 03:26 | 481元 | 无票 |
| **G45** | 北京南 09:56 → 南京南 13:20 | 03:24 | 532元 | 851元 |
| **G9** | 北京南 10:00 → 南京南 13:24 | 03:24 | 532元 | 851元 |
| **G11** | 北京南 10:03 → 南京南 13:28 | 03:25 | 532元 | 851元 |
| G47 | 北京南 10:56 → 南京南 14:27 | 03:31 | 532元 | 851元 |
| G23 | 北京南 11:00 → 南京南 14:31 | 03:31 | 532元 | 851元 |

## 午